# ST-GNN spatial ablations — Infiltration, original labels

Trains **ST Uniform** (two message-passing layers with equal weights for incoming messages) and **ST No Message** (two nodewise transformations without inter-node messages). Both retain the incoming/outgoing feature construction, per-node GRU, and classifier of `ST_GNN_Identity`. The first variant tests learned weighting against uniform aggregation; the second tests neighbor propagation. `Edge GRU` remains an independent baseline, not an identical ablation.

**Original training protocol:** seeds `42, 123, 777, 2024, 99`; at most 60 epochs; Adam with learning rate 0.005; `pos_weight=2`; TBPTT over 10 windows; patience 10 based on validation AP; validation max-F1 threshold; champion chosen by the one-standard-deviation rule on validation F1 and earliest stopping epoch. Test 1 evaluates all five seeds; Test 2 evaluates only the validation-selected champion. Original graphs and labels are used, with no MITRE mapping.

Run cells in order in Colab with a GPU. New results are stored in a separate Drive directory after each completed seed. Rerunning the training cell verifies and skips completed seeds; an interrupted seed starts again. `results_earlystopping` is not modified. Comparisons with the five original models read CSVs already audited by checkpoint inference. If all ten seed records are complete, rerunning the training cell does **not** retrain them.


In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path

if importlib.util.find_spec('torch_geometric') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/nids-mitre')
CLONE_DIR = Path('/content/temporalgnn-nids-st-ablation')
SOURCE_BRANCH = 'feat/fair-comparison-experiments'
SOURCE_COMMIT = '2262b53d155f9e8cf5221516f47e4170e6cd0d6e'
if not CLONE_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_BRANCH,
                    'https://github.com/tatipar/temporalgnn-nids.git', str(CLONE_DIR)], check=True)
actual_commit = subprocess.check_output(
    ['git', '-C', str(CLONE_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_commit == SOURCE_COMMIT, (
    f'Code commit changed: {actual_commit}; expected {SOURCE_COMMIT}'
)
sys.path.insert(0, str(CLONE_DIR / 'code/python'))
print('Pinned code:', actual_commit)

In [ ]:
import gc, hashlib, json, os, platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch_geometric
from sklearn.metrics import average_precision_score
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import softmax as pyg_softmax
from IPython.display import display

from utils.datasets import NF_IDS_Dataset
from utils.models import ST_GNN_Identity
from utils.training import train_epoch, evaluate, find_optimal_threshold, set_seed
from utils.experiment import EarlyStopping, NumpyEncoder
from utils.metrics import calculate_metrics_gnn
from utils.evaluation import apply_1sd_rule

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = [42, 123, 777, 2024, 99]
PARAMS = dict(node_dim=16, edge_dim=32, hidden_dim=32,
              dropout=0.2, output_bias_init=-2.9968)
TRAINING = dict(epochs=60, batch_steps=10, learning_rate=0.005,
                pos_weight=2.0, patience=10, min_delta=0.0001)

DAY1_ROOT = DRIVE_ROOT / 'dataset_processed'
DAY2_ROOT = DRIVE_ROOT / 'dataset_processed_thu0103'
ORIGINAL_RESULTS = DRIVE_ROOT / 'results_earlystopping'
OUTPUT = DRIVE_ROOT / 'results_stgnn_spatial_ablation_v1'
for path in [DAY1_ROOT / 'train', DAY1_ROOT / 'val', DAY1_ROOT / 'test',
             DAY2_ROOT / 'test2', ORIGINAL_RESULTS]:
    assert path.is_dir(), f'Missing: {path}'

train_data = NF_IDS_Dataset(root_dir=str(DAY1_ROOT), split='train')
val_data = NF_IDS_Dataset(root_dir=str(DAY1_ROOT), split='val')
test1_data = NF_IDS_Dataset(root_dir=str(DAY1_ROOT), split='test')
test2_data = NF_IDS_Dataset(root_dir=str(DAY2_ROOT), split='test2')
assert len(test1_data) == 429, f'Test 1: expected 429 graphs; found {len(test1_data)}'
assert len(test2_data) == 2822, f'Test 2: expected 2,822 graphs; found {len(test2_data)}'

# Temporal order and batch_size=1 are required for per-node memory.
loader_opts = dict(batch_size=1, shuffle=False, num_workers=2,
                   persistent_workers=True, pin_memory=False)
train_loader = DataLoader(train_data, **loader_opts)
val_loader = DataLoader(val_data, **loader_opts)
test1_loader = DataLoader(test1_data, **loader_opts)
test2_loader = DataLoader(test2_data, **loader_opts)

manifest = dict(notebook_version=1, source_branch=SOURCE_BRANCH,
                source_commit=SOURCE_COMMIT, graph_train=str(DAY1_ROOT / 'train'),
                graph_val=str(DAY1_ROOT / 'val'), graph_test1=str(DAY1_ROOT / 'test'),
                graph_test2=str(DAY2_ROOT / 'test2'), seeds=SEEDS,
                model_params=PARAMS, training=TRAINING,
                labels='original', champion_rule='validation_F1_1SD_then_min_best_epoch')
OUTPUT.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT / 'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, (
        'Output directory has a different configuration: ' + str(OUTPUT)
    )
else:
    manifest_path.write_text(json.dumps(manifest, indent=2))

print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'none')
print('PyTorch:', torch.__version__, '| PyG:', torch_geometric.__version__)
print('Graphs: train', len(train_data), '| val', len(val_data),
      '| test1', len(test1_data), '| test2', len(test2_data))
print('New results:', OUTPUT)

## 1. Controlled variants

`UniformGATv2Conv` retains GATv2 transformations and connections, including self-loops, but assigns weight `1/k` to every incoming message at a node. In that layer, `edge_attr` no longer determines message weight; it still enters node-feature construction and the final classifier. `STNoMessage` replaces both propagation layers with independent nodewise transformations. Both variants are trained from scratch on their first run.


In [ ]:
class UniformGATv2Conv(GATv2Conv):
    def edge_update(self, x_j, x_i, edge_attr, index, ptr, dim_size):
        zeros = x_j.new_zeros((x_j.size(0), self.heads))
        return pyg_softmax(zeros, index, ptr, dim_size)


class NodewiseLinear(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_attr=None):
        return self.linear(x)


class STUniform(ST_GNN_Identity):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2,
                 output_bias_init=None):
        super().__init__(node_dim, edge_dim, hidden_dim, dropout, output_bias_init)
        first = UniformGATv2Conv(2 * node_dim, hidden_dim, edge_dim=edge_dim,
                                 heads=2, concat=False, dropout=dropout)
        second = UniformGATv2Conv(hidden_dim, hidden_dim, edge_dim=edge_dim,
                                  heads=1, concat=False, dropout=dropout)
        # Preserve initialization of the base-class transformations.
        first.load_state_dict(self.gnn1.state_dict(), strict=True)
        second.load_state_dict(self.gnn2.state_dict(), strict=True)
        self.gnn1, self.gnn2 = first, second
        for layer in (self.gnn1, self.gnn2):
            for name, param in layer.named_parameters():
                if name == 'att' or name.startswith(('lin_r.', 'lin_edge.')):
                    param.requires_grad_(False)  # inactive under uniform weighting


class STNoMessage(ST_GNN_Identity):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2,
                 output_bias_init=None):
        super().__init__(node_dim, edge_dim, hidden_dim, dropout, output_bias_init)
        self.gnn1 = NodewiseLinear(2 * node_dim, hidden_dim)
        self.gnn2 = NodewiseLinear(hidden_dim, hidden_dim)


VARIANTS = {'ST Uniform': STUniform, 'ST No Message': STNoMessage}
for name, cls in VARIANTS.items():
    set_seed(42)
    model = cls(**PARAMS)
    print(name, '| trainable parameters:',
          sum(p.numel() for p in model.parameters() if p.requires_grad))
    del model

## 2. Training and validation across five seeds

This cell can take time when seed records are missing. After **each completed seed**, it saves the checkpoint, training history, best epoch, validation metrics, and max-F1 threshold. On rerun, it verifies and skips completed seeds; an interrupted seed is trained again from the beginning. Champion selection happens later using validation only.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def record_paths(variant, seed):
    stem = variant.lower().replace(' ', '_')
    folder = OUTPUT / stem / f'seed_{seed}'
    return folder, folder / 'checkpoint.pth', folder / 'record.json'


def read_record(variant, seed):
    _, checkpoint, record_path = record_paths(variant, seed)
    if not (checkpoint.exists() and record_path.exists()):
        return None
    record = json.loads(record_path.read_text())
    assert record['variant'] == variant and record['seed'] == seed
    assert record['source_commit'] == SOURCE_COMMIT
    assert record['checkpoint_sha256'] == sha256_file(checkpoint), (
        f'Modified or incomplete checkpoint: {checkpoint}'
    )
    return record


def train_one(variant, seed):
    previous = read_record(variant, seed)
    if previous is not None:
        print(f'{variant}, seed {seed}: already complete; skipping')
        return previous

    folder, checkpoint, record_path = record_paths(variant, seed)
    folder.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    model = VARIANTS[variant](**PARAMS).to(DEVICE)
    optimizer = torch.optim.Adam(
        (p for p in model.parameters() if p.requires_grad),
        lr=TRAINING['learning_rate']
    )
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([TRAINING['pos_weight']], device=DEVICE)
    )
    stopper = EarlyStopping(patience=TRAINING['patience'],
                            min_delta=TRAINING['min_delta'], mode='max')
    history = []
    print(f'Training {variant}, seed {seed}')
    for epoch in range(TRAINING['epochs']):
        train_loss = train_epoch(
            model, train_loader, optimizer, criterion, DEVICE,
            is_temporal=True, batch_steps=TRAINING['batch_steps']
        )
        val_loss, val_y, val_p = evaluate(
            model, val_loader, criterion, DEVICE, is_temporal=True
        )
        val_ap = float(average_precision_score(val_y, val_p))
        assert np.isfinite(val_ap), f'Invalid AP for {variant}, seed {seed}'
        improved = stopper(val_ap, model, epoch)
        history.append(dict(epoch=epoch + 1, train_loss=float(train_loss),
                            val_loss=float(val_loss), val_ap=val_ap))
        if improved or (epoch + 1) % 10 == 0:
            print(f'  epoch {epoch + 1}: train loss={train_loss:.4f}, '
                  f'val AP={val_ap:.4f}' + (' *' if improved else ''))
        if stopper.early_stop:
            print('  early stopping at epoch', epoch + 1)
            break

    assert stopper.best_model_state is not None
    model.load_state_dict(stopper.best_model_state, strict=True)
    threshold, val_y, val_p = find_optimal_threshold(
        model, val_loader, DEVICE, is_temporal=True, strategy='max_f1'
    )
    val_metrics = calculate_metrics_gnn(val_y, val_p,
                                        prob_threshold=float(threshold))
    record = dict(variant=variant, seed=seed,
                  run_id=f'{variant.lower().replace(" ", "_")}_seed{seed}',
                  source_commit=SOURCE_COMMIT,
                  best_epoch=int(stopper.best_epoch),  # same zero-based convention as the project
                  stopped_epoch=len(history),
                  best_val_ap=float(stopper.best_score),
                  threshold=float(threshold), val_metrics=val_metrics,
                  history=history,
                  torch_version=torch.__version__,
                  pyg_version=torch_geometric.__version__)

    # The JSON marks a completed seed; write it last.
    tmp_checkpoint = folder / 'checkpoint.pth.tmp'
    torch.save(model.state_dict(), tmp_checkpoint)
    os.replace(tmp_checkpoint, checkpoint)
    record['checkpoint_sha256'] = sha256_file(checkpoint)
    tmp_record = folder / 'record.json.tmp'
    tmp_record.write_text(json.dumps(record, cls=NumpyEncoder, indent=2))
    os.replace(tmp_record, record_path)
    print(f'  complete: best epoch={record["best_epoch"] + 1}, '
          f'val AP={val_metrics["AUC-PR"]:.4f}, '
          f'val F1={val_metrics["F1"]:.4f}, '
          f'threshold={float(threshold):.6f}')
    del model, optimizer, criterion, stopper
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return record


for variant in VARIANTS:
    for seed in SEEDS:
        train_one(variant, seed)
print('All ten model-seed combinations are complete.')

## 3. Champions selected by validation only

Uses `apply_1sd_rule()` from the same commit as the original evaluation. Among seeds within one standard deviation of the maximum validation F1, it selects the earliest `best_epoch`; ties favor higher F1. Neither Test 1 nor Test 2 enters champion selection.


In [ ]:
validation_rows = []
for variant in VARIANTS:
    for seed in SEEDS:
        record = read_record(variant, seed)
        assert record is not None, f'Missing {variant}, seed {seed}'
        validation_rows.append({
            'Architecture': variant,
            'Raw_Dir_Name': variant,
            'Seed': seed,
            'F1_Score': record['val_metrics']['F1'],
            'Best_Epoch': record['best_epoch'],
            'Optimal_Threshold': record['threshold'],
            'Val_AUC-PR': record['val_metrics']['AUC-PR'],
        })
validation = pd.DataFrame(validation_rows)
champions = apply_1sd_rule(validation)
assert len(champions) == len(VARIANTS)
champion_seeds = dict(zip(champions['Architecture'],
                          champions['Seed'].astype(int)))
validation.to_csv(OUTPUT / 'validation_all_seeds.csv', index=False)
champions.to_csv(OUTPUT / 'champions_by_validation.csv', index=False)
print('Selected seeds:', champion_seeds)
display(validation.sort_values(['Architecture', 'Seed']).round(4))

## 4. Test 1: all five seeds of each variant

Evaluates frozen checkpoints at their validation thresholds. Each completed row is saved to support resuming. Uses the original `evaluate()` and `calculate_metrics_gnn()`, including the strict `probability > threshold` decision rule.


In [ ]:
def evaluate_new(tag, variant, seed):
    assert tag in ('test1', 'test2')
    record = read_record(variant, seed)
    assert record is not None
    folder, checkpoint, _ = record_paths(variant, seed)
    cache = folder / f'{tag}_metrics.json'
    if cache.exists():
        row = json.loads(cache.read_text())
        assert row['checkpoint_sha256'] == record['checkpoint_sha256']
        assert row['threshold'] == record['threshold']
        print(f'{tag}: {variant}, seed {seed}: already evaluated')
        return row

    model = VARIANTS[variant](**PARAMS).to(DEVICE)
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE,
                                     weights_only=True), strict=True)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([TRAINING['pos_weight']], device=DEVICE)
    )
    loader = test1_loader if tag == 'test1' else test2_loader
    _, labels, probabilities = evaluate(
        model, loader, criterion, DEVICE, is_temporal=True
    )
    if tag == 'test2':
        assert len(labels) == 2_135_957, (
            f'Test 2 should have 2,135,957 flows; found {len(labels)}'
        )
    metrics = calculate_metrics_gnn(
        labels, probabilities, prob_threshold=record['threshold']
    )
    row = dict(test=tag, model=variant, seed=seed,
               run_id=record['run_id'],
               threshold=record['threshold'],
               flows=len(labels), attacks=int(labels.sum()),
               checkpoint_sha256=record['checkpoint_sha256'], **metrics)
    tmp = folder / f'{tag}_metrics.json.tmp'
    tmp.write_text(json.dumps(row, cls=NumpyEncoder, indent=2))
    os.replace(tmp, cache)
    print(f'{tag}: {variant}, seed {seed}: '
          f'AP={metrics["AUC-PR"]:.4f}, F1={metrics["F1"]:.4f}')
    del model, criterion
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return row


new_t1 = pd.DataFrame([
    evaluate_new('test1', variant, seed)
    for variant in VARIANTS for seed in SEEDS
])
assert len(new_t1) == 10
assert new_t1['flows'].nunique() == new_t1['attacks'].nunique() == 1
new_t1.to_csv(OUTPUT / 'test1_all_seeds.csv', index=False)
display(new_t1[['model', 'seed', 'flows', 'attacks',
                'AUC-PR', 'F1', 'Precision', 'Recall', 'FPR']].round(4))

## 5. Test 2: validation-selected champions only

Validation has already fixed checkpoints and thresholds. Each variant is evaluated once on the 2,135,957 Test 2 flows; the best seed is not selected using Test 2.


In [ ]:
new_t2 = pd.DataFrame([
    evaluate_new('test2', variant, champion_seeds[variant])
    for variant in VARIANTS
])
assert len(new_t2) == 2
new_t2.to_csv(OUTPUT / 'test2_champions.csv', index=False)
display(new_t2[['model', 'seed', 'flows', 'attacks',
                'AUC-PR', 'F1', 'Precision', 'Recall', 'FPR', 'TP', 'FP']].round(4))

## 6. Comparison with audited original models

Reads the audited original CSVs for Test 1 (five seeds) and Test 2 (one champion per architecture). New results are joined only if flow and attack counts match. Test 1 reports mean and standard deviation across seeds; Test 2 reports each champion. Paired same-seed differences against ST-GNN and Edge GRU are also shown for Test 1.


In [ ]:
ORIGINAL_MODELS = {
    'Simple MLP': 'SimpleMLP_BiasOn',
    'E-GraphSAGE': 'EGraphSAGE_BiasOn',
    'Edge GRU': 'EdgeGRU_NoX_BiasOn',
    'Static GNN': 'StaticGNN_BiasOn_robust_Identity',
    'ST-GNN': 'ST_GNN_BiasOn_robust_Identity_clone',
}

def original_rows(tag):
    parts = []
    for display_name, experiment in ORIGINAL_MODELS.items():
        if tag == 'test1':
            path = ORIGINAL_RESULTS / 'test1_results' / f'test1_metrics_{experiment}.csv'
            expected_rows = 5
        else:
            path = ORIGINAL_RESULTS / 'test2_results' / f'test2_metrics_{experiment}.csv'
            expected_rows = 1
        frame = pd.read_csv(path)
        assert len(frame) == expected_rows, f'Unexpected row count: {path}'
        assert frame['run_id'].notna().all(), f'Missing run_id: {path}'
        assert frame['run_id'].is_unique, f'Duplicate run_id: {path}'
        if tag == 'test1':
            assert set(frame['seed'].astype(int)) == set(SEEDS), path
        frame['model'] = display_name
        frame['seed'] = frame['seed'].astype(int)
        frame['flows'] = frame[['TP', 'FP', 'TN', 'FN']].sum(axis=1).astype(int)
        frame['attacks'] = (frame['TP'] + frame['FN']).astype(int)
        parts.append(frame)
    return pd.concat(parts, ignore_index=True)

orig_t1 = original_rows('test1')
orig_t2 = original_rows('test2')
assert orig_t1['flows'].nunique() == orig_t1['attacks'].nunique() == 1
assert orig_t2['flows'].nunique() == orig_t2['attacks'].nunique() == 1
assert int(orig_t1['flows'].iloc[0]) == int(new_t1['flows'].iloc[0])
assert int(orig_t1['attacks'].iloc[0]) == int(new_t1['attacks'].iloc[0])
assert int(orig_t2['flows'].iloc[0]) == int(new_t2['flows'].iloc[0])
assert int(orig_t2['attacks'].iloc[0]) == int(new_t2['attacks'].iloc[0])

columns = ['model', 'seed', 'flows', 'attacks', 'AUC-PR', 'F1',
           'Precision', 'Recall', 'FPR', 'TP', 'FP']
all_t1 = pd.concat([orig_t1[columns], new_t1[columns]], ignore_index=True)
all_t2 = pd.concat([orig_t2[columns], new_t2[columns]], ignore_index=True)
order = list(ORIGINAL_MODELS) + list(VARIANTS)
all_t1['model'] = pd.Categorical(all_t1['model'], categories=order, ordered=True)
all_t2['model'] = pd.Categorical(all_t2['model'], categories=order, ordered=True)
all_t1 = all_t1.sort_values(['model', 'seed'])
all_t2 = all_t2.sort_values('model')
all_t1.to_csv(OUTPUT / 'comparison_test1_all_seeds.csv', index=False)
all_t2.to_csv(OUTPUT / 'comparison_test2_champions.csv', index=False)

summary_t1 = all_t1.groupby('model', observed=True).agg(
    AP_mean=('AUC-PR', 'mean'), AP_std=('AUC-PR', 'std'),
    F1_mean=('F1', 'mean'), F1_std=('F1', 'std'),
    Recall_mean=('Recall', 'mean'), FPR_mean=('FPR', 'mean'),
    seeds=('seed', 'count')
).reset_index()
summary_t1.to_csv(OUTPUT / 'comparison_test1_summary.csv', index=False)
print('TEST 1 — five seeds:')
display(summary_t1.round(4))
print('TEST 2 — validation-selected champion:')
display(all_t2[['model', 'seed', 'AUC-PR', 'F1', 'Precision',
                'Recall', 'FPR', 'TP', 'FP']].round(4))

paired_rows = []
for variant in VARIANTS:
    for reference in ['ST-GNN', 'Edge GRU']:
        left = all_t1.loc[all_t1['model'] == variant,
                          ['seed', 'AUC-PR', 'F1']]
        right = all_t1.loc[all_t1['model'] == reference,
                           ['seed', 'AUC-PR', 'F1']]
        pair = left.merge(right, on='seed', suffixes=('_new', '_reference'),
                          validate='one_to_one')
        assert len(pair) == 5
        for _, row in pair.iterrows():
            paired_rows.append(dict(variant=variant, reference=reference,
                                    seed=int(row['seed']),
                                    AP_difference=row['AUC-PR_new'] - row['AUC-PR_reference'],
                                    F1_difference=row['F1_new'] - row['F1_reference']))
paired = pd.DataFrame(paired_rows)
paired.to_csv(OUTPUT / 'comparison_test1_paired_differences.csv', index=False)
print('Same-seed differences on Test 1 (variant minus reference):')
display(paired.round(4))
print('Comparison files:', OUTPUT)

## Methodological interpretation

- **ST Uniform vs. ST-GNN:** retrained models with and without adaptive message attention, retaining graph connections and the GRU. `edge_attr` contributes to attention in the original GAT; uniform weighting removes that contribution to attention, while leaving it in node attributes and the classifier.
- **ST Uniform vs. ST No Message:** tests neighbor-representation propagation with the rest of the ST design retained. The architectures have different numbers of active parameters, which the notebook reports.
- **ST No Message vs. Edge GRU:** both are temporal models without GAT propagation, but use different input summaries and classifiers. Their difference cannot be assigned to a single component.
- **Test 2:** one champion per architecture. Gains concentrated in a few sources require further analysis before generalization. No variant evaluates attack stages or chains.
